Summary:
1. For thresholds, 3 columns - total_counts, n_genes, and mito_pct has been considered.
    a. donors that fail all three were dropped (2 donors)
    b. donors that fail some were flagged (10 donors)
    c. separately, donors with fewer than 100 passing cells were dropped (10 donors, all Healthy)
2. Unscaled MAD with k=5 has been applied to both donor and cells.
3. For mito_pct, only upper side is considered.
4. 330 of 342 donors kept (CD inflamed 62/63, CD uninflamed 46/47, Healthy 222/232); 
5. 269,013 of 292,869 T cells pass (91.9%); failure rate is similar across groups (7.2–8.4%)
6. In Eterocyte cells, CD inflamed has a slight bump in mito at 40-50%, on further analysis, it seems to arise mostly from one donor. Enterocyte QC is postponed to the enterocyte stage. This donor's T cells are normal, so it is kept for the T cell analysis
7. In T cells, CD4- CD8- contribute to a 68% of failing cells - either no cytoplasm or damanged cells 

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [ ]:
obs = pd.read_parquet(r'..\data\processed\obs_clean.parquet')

In [ ]:
obs.shape

In [ ]:
obs.head()

Let us look into mitochondrial rna percentage.  
When the cell is damaged, the cell rna leaks out, while the share of mito rna stays inside and hence goes up.  
We will check for cells with high percentage of this mitochondrial rna, otherwise all the stressed/damaged cells will show appear co-expressed

'pct_counts_gene_group__mito_transcript' column refers to the percentage of the cell rna that belongs to one of the 13 mitochondrial genes

In [ ]:
t_obs = obs[obs['category']=='T cell'].copy()
e_obs = obs[obs['category']=='Enterocyte'].copy()

In [ ]:
cell_types = ['T cell', 'Enterocyte']
order = ['Healthy', 'CD uninflamed', 'CD inflamed']
cell_obs = {"T cell": t_obs, "Enterocyte": e_obs}

In [ ]:
mito = 'pct_counts_gene_group__mito_transcript'

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=True)

for ax, (name, df) in zip(axes, cell_obs.items()):
    values = df[mito]
    ax.hist(values, bins=100)
    ax.set_yscale('log')
    ax.axvline(values.median(), color='red', ls='--', label=f'median ({values.median():.1f}%)')
    ax.axvline(values.mean(), color='black', ls=':', label=f'mean ({values.mean():.1f}%)')
    ax.set_title(name)
    ax.set_xlabel(f'mito % in {name}s')
    ax.legend()

axes[0].set_ylabel('No of cells (log scale)')
plt.tight_layout()
plt.show()


50% of T cells has about 5-6% of mitochondrial RNA, whereas that share ranges between 18-20% for Enterocytes.  
This might be because they are energy hungry cells.  

In [ ]:
t_obs.groupby('health_category')['pct_counts_gene_group__mito_transcript'].quantile([0.5, 0.9, 0.95, 0.99]).unstack()


99% of the cells have less than 49-54% mito rna

In [ ]:
e_obs.groupby('health_category')['pct_counts_gene_group__mito_transcript'].quantile([0.5, 0.9, 0.95, 0.99]).unstack()

99% of the cells have less than 70% mito rna

We will be using density instead of counts to see the shape. this is because cells from healthy donor are more in number, and will put the healthy category curve higher than the other

In [ ]:
mito = 'pct_counts_gene_group__mito_transcript'

fig, axes = plt.subplots(1, len(cell_types), figsize=(7 * len(cell_types), 5), sharey=True)

for ax, (name, df) in zip(axes, cell_obs.items()):
    sns.histplot(data=df, x=mito,
                hue='health_category', hue_order=order, palette='Set2',
                bins=np.arange(0, 81, 1), stat='density', common_norm=False,
                element='poly', fill=False, ax=ax)
    ax.set_title(name)
    ax.set_xlabel('Mitochondrial transcripts (% of counts)')
    ax.set_yscale('log')

axes[0].set_ylabel('Density (log scale)')
plt.tight_layout()
plt.show()


Observations
1. For both T cell and Enterocytes, there is a spike near 0% before dipping down and going back up for the main peak. Are they just low data cells?
2. There is a steady decline in the density of cells as the mito% goes up. Cutoff for both differs and will be decided based on further evaluation
3. In enterecoytes, there is a little spike between 40 - 50% only for CD inflamed. Real? (damaged/stressed cells in inflamed gut) or technical?

Comparing total counts and n_genes_by_counts with low mito % for T cells

In [ ]:
group_order = ['exactly 0', '0-1%', '1% or more']
colors = {'exactly 0': 'tab:red', '0-1%': 'tab:orange', '1% or more': 'tab:gray'}
metrics = {'total_counts': 'Total counts (log scale)',
        'n_genes_by_counts': 'Genes detected (log scale)'}

fig, axes = plt.subplots(len(metrics), len(cell_types),
                         figsize=(7 * len(cell_types), 4.5 * len(metrics)), sharey='row')

for row, (metric, label) in zip(axes, metrics.items()):
    for ax, (ct, c_obs) in zip(row, cell_obs.items()):
        groups = np.select([c_obs[mito] == 0, c_obs[mito] < 1],
                        ['exactly 0', '0-1%'], default='1% or more')
        sns.histplot(data=c_obs.assign(mito_group=groups), x=metric,
                    hue='mito_group', hue_order=group_order, palette=colors,
                    bins=60, log_scale=True, stat='density', common_norm=False,
                    element='step', fill=False, ax=ax)
        ax.set_title(ct)
        ax.set_xlabel(label)
    row[0].set_ylabel('Density')

plt.tight_layout()
plt.show()


In [ ]:
group_order = ['exactly 0', '0-1%', '1% or more']
rows = []

for ct, c_obs in cell_obs.items():
    groups = pd.Series(
        np.select([c_obs[mito] == 0, c_obs[mito] < 1], ['exactly 0', '0-1%'], default='1% or more'),
        index=c_obs.index, name='mito_group')
    t = c_obs.groupby(groups).agg(n_cells=('total_counts', 'size'),
        median_counts=('total_counts', 'median'),
        median_genes=('n_genes_by_counts', 'median'))
    t['pct_of_cells'] = (100 * t['n_cells'] / t['n_cells'].sum()).round(1)
    t.insert(0, 'cell_type', ct)
    rows.append(t.reindex(group_order))

summary = pd.concat(rows).reset_index().set_index(['cell_type', 'mito_group'])
summary[['n_cells', 'pct_of_cells', 'median_counts', 'median_genes']]


Observations
1. Most of the spike near 0% mito is low-data cells. In T cells, cells with exactly 0% mito have a median of about 1,000 counts and 520 genes, against about 4,500 counts and 2,040 genes for cells with 1% or more.
2. T cells show a gradient: exactly 0% < 0-1% < 1% or more, in both total counts and genes.
3. Enterocytes show a jump instead: the two low-mito groups are almost the same (about 1,900-2,000 counts, about 1,100 genes) and far below the rest (about 22,000 counts, about 5,000 genes).
4. The low-mito groups are small
5. In the low-mito groups there are only about 1.7-2 molecules per detected gene, against 2.2 (T cells) and 4.5 (Enterocytes) in the rest, so there is very little data per gene.

Kept as a separate decision. Most are caught by the low genes/counts rules, and they are concentrated in the DN cluster

### Is the 40-50% bump in CD inflamed enterocytes real or technical?
Idea: if it is real biology, many CD inflamed donors should contribute cells to it. If it is technical, a few donors (or one sample) will hold most of it.  
Donors, not cells, are our unit, so we look at how the bump cells are spread over donors.

In [ ]:
# the bump: enterocytes with 40-50% mito, looked at in CD inflamed donors only
bump_low, bump_high = 40, 50

ci = e_obs[e_obs['health_category'] == 'CD inflamed']          # all CD inflamed enterocytes
e_bump = ci[ci[mito].between(bump_low, bump_high)]             # the bump cells

print(f"CD inflamed donors with enterocytes: {ci['individual_id'].nunique()}")
print(f"donors with at least one bump cell : {e_bump['individual_id'].nunique()}")
print(f"bump cells                         : {len(e_bump)}")

Almost every donor (62 of 63) has at least one bump cell, so counting donors tells us nothing. What matters is how many cells each donor gives.

In [ ]:
# how many bump cells does each donor contribute?
per_donor = e_bump['individual_id'].value_counts()
share = per_donor / per_donor.sum()

pd.DataFrame({'bump_cells': per_donor, 'share': share, 'cumulative_share': share.cumsum()}).head(10).round(3)

One donor holds 47% of all bump cells. The next donor holds only 5%, and the top 3 hold 56%. So the bump is not spread evenly.

In [ ]:
# what fraction of each donor's own enterocytes sit in the bump?
frac = (e_bump.groupby('individual_id').size() / ci.groupby('individual_id').size()).fillna(0)
frac.sort_values(ascending=False).head(10).round(3)

For the top donor, 33% of its own enterocytes sit in the bump. For everyone else it is 11% or less, so this donor looks very different from the rest.

In [ ]:
# the top donor, defined once and reused below
donor = per_donor.index[0]
donor_obs = obs[obs['individual_id'] == donor]

print(f"{donor} | health: {donor_obs['health_category'].iloc[0]} | "
      f"samples: {donor_obs['sanger_sample_id'].nunique()} | cells: {len(donor_obs)}")

In [ ]:
# mito % of this donor's enterocytes vs enterocytes of the other CD inflamed donors
plot_df = ci.assign(who=np.where(ci['individual_id'] == donor, 'this donor', 'other CD inflamed donors'))

ax = sns.histplot(data=plot_df, x=mito, hue='who', bins=np.arange(0, 81, 2),
                  stat='density', common_norm=False, element='step', fill=False)
sns.move_legend(ax, 'upper left', bbox_to_anchor=(1, 1))
ax.set_xlabel('Mitochondrial transcripts (% of counts)')
ax.set_title('Enterocytes, CD inflamed')
plt.tight_layout()
plt.show()

This donor's enterocytes have their own peak at about 45-50% mito, while the other CD inflamed donors peak around 15-20%. The donor also has a group near 0% and a group around 15-20%, so it is a mix, with a separate damaged-looking group on top.

In [ ]:
# is it only enterocytes, or the whole sample? median mito per cell type, this donor vs everyone else
others_obs = obs[obs['individual_id'] != donor]

pd.DataFrame({
    'n_cells_this_donor': donor_obs.groupby('category').size(),
    'mito_this_donor': donor_obs.groupby('category')[mito].median(),
    'mito_other_donors': others_obs.groupby('category')[mito].median(),
    'counts_this_donor': donor_obs.groupby('category')['total_counts'].median(),
    'counts_other_donors': others_obs.groupby('category')['total_counts'].median(),
}).round(1)

Not only enterocytes, but plasma B cells are also very high (39% vs 2% in other donors) with very low counts. T cells, myeloid, B cells, secretory and stem cells look roughly normal. So the damage is selective

In [ ]:
# technical setup of this donor, and whether other donors with the same setup look normal
tech = ['bead_version', 'chip_version', 'bead_lot', 'chip_lot', 'gem_lot', 'enzyme_lot_BLP', 'f2_f1_ratio']

# one row per donor
donors = obs.groupby('individual_id')[['health_category'] + tech].first()
donors['ent_mito'] = e_obs.groupby('individual_id')[mito].median()
others = donors.drop(donor)

rows = []
for col in tech:
    same = others[others[col] == donors.loc[donor, col]]      # other donors with the same value
    rows.append({'factor': col,
                'value': donors.loc[donor, col],
                'other_donors_sharing': len(same),
                'of_which_CD_inflamed': (same['health_category'] == 'CD inflamed').sum(),
                'their_median_ent_mito': same['ent_mito'].median()})

pd.DataFrame(rows).set_index('factor').round(1)


Other donors with the same bead, chip, gem or enzyme lot have normal enterocyte mito (about 15-20%), and so do the 16 other V3/chip-b donors (17%). So a bad reagent lot is unlikely; the problem seems specific to this sample.  
f2_f1 6:1 has only one other donor, so there is nothing to compare.

In [ ]:
# leave-one-out: does the bump stay if we remove this donor?
e_all = e_obs.assign(in_bump=e_obs[mito].between(bump_low, bump_high))

with_donor    = 100 * e_all.groupby('health_category')['in_bump'].mean()
without_donor = 100 * e_all[e_all['individual_id'] != donor].groupby('health_category')['in_bump'].mean()

print(pd.DataFrame({'with donor (%)': with_donor, 'without donor (%)': without_donor}).loc[order].round(2))

Without this donor, the bump in CD inflamed falls from 5.3% to 3.0%, in line with Healthy (3.7%) and CD uninflamed (3.9%)

The bump in CD enterocytes is mostly one donor, and the reagents lot does not explain it, since other donors on the same lots are normal.  
We flag this donor but do not drop it. We will check later in PCA analysis

Let us look into the total counts distribution.  
Total counts tell us how many rna molecules was found in a cell while sequencing

In [ ]:
fig, axes = plt.subplots(1, len(cell_types), sharey=True, figsize=(7 * len(cell_types), 5))
for ax, (ct, c_obs)  in zip(axes, cell_obs.items()):
    sns.histplot(data=c_obs, x='total_counts', bins=100, ax=ax, hue='health_category', 
                palette='Set2', element='poly', log_scale=True, stat='density', common_norm=False,
                fill=False, hue_order=order)
    ax.set_title(ct)
    ax.set_xlabel('Total counts (UMIs)')
    ax.set_yscale('log')
axes[0].set_ylabel('Density (log scale)')
plt.tight_layout()
plt.show()

Observations.
1. T cells have a tail on left with total counts less than 100 - low data cells
2. T cells main peak is at 6500 approximately
3. Enterocytes main peak is at around 35k approx, and first peak is at round 1500 approx
4. Enterocytes scale runs to 10^6 whereas T cells runs till 10^5, about 10 times higher

In [ ]:
t_obs.groupby('health_category')['total_counts'].quantile([0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).unstack()


In [ ]:
e_obs.groupby('health_category')['total_counts'].quantile([0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).unstack()


Observations
1. Total counts: the T cell median is about 4,100-4,900 and the Enterocyte median is about 21,000-23,000, roughly 5 times higher.
2. In both cell types CD inflamed has slightly higher counts than Healthy (e.g. Enterocyte 99% quantile: 153k vs 111k). These differences are small compared to the difference between the two cell types.
3. The Enterocyte 25% quantile is lower in CD inflamed (6,410) than in the other groups (8,800-9,700), so CD inflamed has more low-count Enterocytes.

Let us look at the distribution of the column 'n_genes_by_counts'.  
It tells us about the unique number of genes found in the cell, whereas total counts tells us all rna molecules found. 
total_counts/n_genes_by_counts tells us the average number of molecules detected per gene.  

In [ ]:
fig, axes = plt.subplots(1, len(cell_types), sharey=True, figsize=(7 * len(cell_types), 5))
for ax, (ct, c_obs)  in zip(axes, cell_obs.items()):
    sns.histplot(data=c_obs, x='n_genes_by_counts', bins=100, ax=ax, hue='health_category', 
                palette='Set2', element='poly', log_scale=True, stat='density', common_norm=False,
                fill=False, hue_order=order)
    ax.set_title(ct)
    ax.set_xlabel('n genes by counts in a cell')
    ax.set_yscale('log')
axes[0].set_ylabel('Density (log scale)')
plt.tight_layout()
plt.show()

In [ ]:
t_obs.groupby('health_category')['n_genes_by_counts'].quantile([0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).unstack()

In [ ]:
e_obs.groupby('health_category')['n_genes_by_counts'].quantile([0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).unstack()

Observations
1. Genes detected: T cells have one peak at about 2,500 genes, Enterocytes peak at about 6,000 with a smaller peak around 1,000 (same two-peak shape as total counts).
2. Medians: T cell about 1,900-2,200 genes, Enterocyte about 4,900-5,100. Enterocytes detect about 2.5 times more genes, less than the 5 times more counts.
3. The three health groups overlap closely. CD inflamed is slightly higher in the middle of the distribution.
4. The wiggles at the far left (below about 100 genes) are only a few cells, so they are noise.

Let us look at the spread of cells over total counts and genes detected, with mito %.  
We randomly select a 30k cells, hence this shows the shape and not counts

In [ ]:
mito = 'pct_counts_gene_group__mito_transcript'

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=True)

for ax, (name, df) in zip(axes, cell_obs.items()):
    sns.scatterplot(data=df.sample(30_000, random_state=42),
                    x='total_counts', y='n_genes_by_counts',
                    hue=mito, palette='viridis', hue_norm=(0, 50),
                    s=4, alpha=0.5, edgecolor=None, legend=False, ax=ax)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_title(name)
    ax.set_xlabel('Total counts')

axes[0].set_ylabel('Genes detected')

sm = plt.cm.ScalarMappable(cmap='viridis', norm=plt.Normalize(0, 50))
fig.colorbar(sm, ax=axes, label='mito % (colors capped at 50)')
plt.show()


After changing the seed around, the shape holds, the position shift very little to left or right

Observations
1. Cells with more total counts also detect more genes, so both cell types follow one curved band.
2. High mito cells (yellow) sit at the bottom edge of the band: for the same total counts they detect fewer genes. This is what damaged cells look like.
3. The yellow band is much clearer in Enterocytes than in T cells.
4. A few Enterocytes lie far below the band (high counts, low mito but only 10-100 genes). These look odd and are worth checking later.

### Per donor Table for T cells

In [ ]:
donor_qc = (
    t_obs.groupby('individual_id')
    .agg(
        health_category=('health_category', 'first'),
        age=('age_binned_10', 'first'),
        sex=('sex', 'first'),

        n_cells=('individual_id', 'size'),
        n_samples=('sanger_sample_id', 'nunique'),

        median_total_counts=('total_counts', 'median'),
        median_n_genes=('n_genes_by_counts', 'median'),
        median_mito=(mito, 'median'),

        pct_mito_zero=(mito, lambda x: (x == 0).mean() * 100),
        pct_mito_above_20=(mito, lambda x: (x > 20).mean() * 100),
    )
    .reset_index()
)

In [ ]:
donor_qc.head()

In [ ]:
def mad_thresholds(col: str, df, ks=(3, 5)):
    """Median +/- k MAD cuts on the log10 scale. Returns one row per k."""
    x = np.log10(df[col] + 1)                  # +1 so zeros (e.g. mito 0%) do not become -inf
    med = x.median()
    mad = (x - med).abs().median()

    print(f"{col} | cells/donor: {len(df)} | median: {10**med - 1:.1f}")
    print(f"MAD (log units): {mad:.2f} -> factor {10**mad:.2f}")
    print(f"min: {df[col].min():.1f} | max: {df[col].max():.1f}")

    rows = []
    for k in ks:
        low, high = 10**(med - k*mad) - 1, 10**(med + k*mad) - 1      # back to the original scale
        below, above = (df[col] < low).sum(), (df[col] > high).sum()
        rows.append({'k_MAD': k, 'lower_cut': low, 'upper_cut': high,
                    'n_below': below, 'n_above': above,
                     'pct_removed': 100 * (below + above) / len(df)})
    return pd.DataFrame(rows).set_index('k_MAD').round({'lower_cut': 0, 'upper_cut': 0, 'pct_removed': 1})


In [ ]:
print('Per cell')
display(mad_thresholds('total_counts', t_obs))
print()
print('Per Donor')
display(mad_thresholds('median_total_counts', donor_qc))

In [ ]:
print('Per cell')
display(mad_thresholds('pct_counts_gene_group__mito_transcript', t_obs))
print()
print('Per Donor')
display(mad_thresholds('median_mito', donor_qc))

For mito% we will only be considering upper limit and now lower

In [ ]:
print('Per cell')
display(mad_thresholds('n_genes_by_counts', t_obs))
print()
print('Per Donor')
display(mad_thresholds('median_n_genes', donor_qc))

MAD is unscaled (no 1.4826). Decided to use 5 MAD since 3 MAD was a little too tight.  
At 3 MAD, each metric alone removes 9–15% of cells (mito 14.5%). At 5 MAD, it's 0.7–3.6%.

In [ ]:
def mad_flag_donor(df, col, k, side='both'):
    """True for rows outside median +/- k MAD (log10 scale). side: 'both', 'low' or 'high'."""
    x = np.log10(df[col] + 1)
    med = x.median()
    mad = (x - med).abs().median()
    low, high = x < med - k*mad, x > med + k*mad
    return {'both': low | high, 'low': low, 'high': high}[side]

k = 5
donor_qc['low_counts'] = mad_flag_donor(donor_qc, 'median_total_counts', k, side='low')
donor_qc['low_genes']  = mad_flag_donor(donor_qc, 'median_n_genes', k, side='low')
donor_qc['high_mito']  = mad_flag_donor(donor_qc, 'median_mito', k, side='high')   # only the upper side matters
flag_cols = ['low_counts', 'low_genes', 'high_mito']
donor_qc['any_flag'] = donor_qc[flag_cols].any(axis=1)

# the flagged donors
donor_qc[donor_qc['any_flag']].sort_values('n_cells')


Let us remove donor IBDverse-1-A98723092B1FB1 and IBDverse-1-F211C1C8D550EC because they have low counts, low genes and high mito  
Everything else will be flagged to watch and decide later

In [ ]:
drop_ids = ['IBDverse-1-A98723092B1FB1', 'IBDverse-1-F211C1C8D550EC']
flag_ids = ['IBDverse-1-78461EAC955C56', 'IBDverse-1-D324F9FF3771BD', 'IBDverse-1-FD5121B50F5FAB', 'IBDverse-1-B2C0164E6311E0', 'IBDverse-1-7197068CD420EC', 'IBDverse-1-D2B593FEDCE62D', 'IBDverse-1-8D29C5BD7C12C7', 'IBDverse-1-3A5C2FF82FA8A4', 'IBDverse-1-FDD4FC929E8E2B', 'IBDverse-1-0BDB4FAA6060DA']


In [ ]:
# donor level: one row per donor
donor_qc['donor_decision'] = 'keep'
donor_qc.loc[donor_qc['individual_id'].isin(drop_ids), 'donor_decision'] = 'drop'
donor_qc.loc[donor_qc['individual_id'].isin(flag_ids), 'donor_decision'] = 'keep_flagged'

In [ ]:
donor_qc['donor_decision'].value_counts()

In [ ]:
# copy the decision onto each cell
t_obs['donor_ok'] = t_obs['individual_id'].map(
    donor_qc.set_index('individual_id')['donor_decision']) != 'drop'

In [ ]:
t_obs['donor_ok'].value_counts()

In [ ]:
def mad_flag_cells(df, col, k=5, side='both'):
    x = np.log10(df[col]+1)
    donor = df['individual_id']
    med = x.groupby(donor).transform('median')
    mad = (x-med).abs().groupby(donor).transform('median')
    low = x < (med - k * mad)
    high = x> (med + k * mad)
    return {'both': low | high, 'low': low, 'high': high}[side]

In [ ]:
k = 5
fail_counts = mad_flag_cells(t_obs, 'total_counts', k, side='both')
fail_genes = mad_flag_cells(t_obs, 'n_genes_by_counts', k, side='both')
fail_mito = mad_flag_cells(t_obs, 'pct_counts_gene_group__mito_transcript', k, side='high')

n_fail = fail_counts.astype(int) + fail_genes.astype(int) + fail_mito.astype(int)
n_fail.value_counts()

Let us check for cells which only belongs to donors which have decent cell data

In [ ]:
fail_counts = mad_flag_cells(t_obs[t_obs['donor_ok']==True], 'total_counts', k, side='both')
fail_genes = mad_flag_cells(t_obs[t_obs['donor_ok']==True], 'n_genes_by_counts', k, side='both')
fail_mito = mad_flag_cells(t_obs[t_obs['donor_ok']==True], 'pct_counts_gene_group__mito_transcript', k, side='high')

n_fail = fail_counts.astype(int) + fail_genes.astype(int) + fail_mito.astype(int)
n_fail.value_counts()

In [ ]:
labels = {0: 'pass', 1: '1 rule', 2: '2 rules', 3: '3 rules'}

counts = n_fail.value_counts().sort_index()
counts.index = counts.index.map(labels)          # index becomes readable labels
counts['any failed'] = counts.drop('pass').sum() # the union row, added in one line

failing = pd.DataFrame({'n_cells': counts, 'pct (%)': (counts / n_fail.size * 100).round(1)})
failing.index.name = 'rules failed'
failing

In [ ]:
t_obs['fail_counts'] = mad_flag_cells(t_obs, 'total_counts', k, side='both')
t_obs['fail_genes']  = mad_flag_cells(t_obs, 'n_genes_by_counts', k, side='both')
t_obs['fail_mito']   = mad_flag_cells(t_obs, mito, k, side='high')

t_obs['cell_ok'] = ~(t_obs['fail_counts'] | t_obs['fail_genes'] | t_obs['fail_mito'])
t_obs['pass_qc'] = t_obs['donor_ok'] & t_obs['cell_ok']

In [ ]:
t_obs['pass_qc'].sum() == t_obs[t_obs['donor_ok']==True]['cell_ok'].sum()

In [ ]:
print(f'failed counts (%): {fail_counts.mean()*100}')
print(f'failed genes (%): {fail_genes.mean()*100}')
print(f'failed mito (%): {fail_mito.mean()*100}')

In [ ]:
low_fail_counts = mad_flag_cells(t_obs[t_obs['donor_ok']==True], 'total_counts', k, side='low')
high_fail_counts = mad_flag_cells(t_obs[t_obs['donor_ok']==True], 'total_counts', k, side='high')
print(f'failed counts low (%): {low_fail_counts.mean()*100}')
print(f'failed counts high (%): {high_fail_counts.mean()*100}')

In [ ]:
low_fail_genes = mad_flag_cells(t_obs[t_obs['donor_ok']==True], 'n_genes_by_counts', k, side='low')
high_fail_genes = mad_flag_cells(t_obs[t_obs['donor_ok']==True], 'n_genes_by_counts', k, side='high')
print(f'failed genes low (%): {low_fail_genes.mean()*100}')
print(f'failed genes high (%): {high_fail_genes.mean()*100}')

Check how many cells are failing in differnt health categories in percentatge

In [ ]:
t_obs[t_obs['donor_ok']==True].groupby('health_category')['cell_ok'].apply(lambda x: (1-x.mean())*100)

There is not muchh difference (1%) between failing cells in different health categories

In [ ]:
t_obs[t_obs['donor_ok']==True].groupby('label')['cell_ok'].apply(lambda x: (1-x.mean())*100)

Since most of the cells failing falls under 3 subtypes, we will check if it is actually bad cells or just a feature of the cell types

In [ ]:
t_obs['zero_mito'] = t_obs['pct_counts_gene_group__mito_transcript'] == 0

In [ ]:
to_investigate = ['T cells CD4- CD8-', 'T cells CD4- CD8-lo', 'T cells CD4+ LEF1+']
d = t_obs[t_obs['donor_ok']]

d.groupby('label', observed=True).agg(
    n=('label', 'size'),
    med_genes=('n_genes_by_counts', 'median'),
    pct_fail_genes=('fail_genes', lambda x: x.mean()*100),
    med_counts=('total_counts', 'median'),
    pct_fail_counts=('fail_counts', lambda x: x.mean()*100),
    med_mito=('pct_counts_gene_group__mito_transcript', 'median'),
    pct_fail_mito=('fail_mito', lambda x: x.mean()*100),
    pct_zero_mito=('zero_mito', lambda x: x.mean()*100),
)

T cells CD4- CD8- has low RNA, both counts and genes. But it also has a high 0% mito and low mito%. The 0% mito and low RNA might suggest empty cells.  
Whereas, T cells CD4- CD8-lo has higher mito percentage and lower genes. This might suggest damaged cells.  
So the threshold removes poor cells, so the QC stays as is.

In [ ]:
(d['label'].isin(to_investigate) & ~d['cell_ok']).sum()/(~d['cell_ok']).sum() * 100

Around 68% of bad cells come from the 3 categories

In [ ]:
pd.crosstab(d['health_category'], d['label'], normalize='index') * 100

The 3 suspicious subtypes make up a similar share of T cells in every group (DN 7.1–8.2%, CD8-lo 4.7–5.6%, LEF1+ 1.2–1.5%), so QC removes them evenly across groups

After removing cells deemed unfit, how many donors will have less than 100 cells?

In [ ]:
n_pass = (t_obs[t_obs['pass_qc']].groupby('individual_id').size().reindex(donor_qc['individual_id'], fill_value=0))
(n_pass<100).sum()

In [ ]:
(n_pass<50).sum()

In [ ]:
low = n_pass[n_pass < 100].index
donor_qc.set_index('individual_id').loc[low, ['health_category', 'donor_decision', 'n_cells']].assign(n_pass=n_pass[low])

We will choose 100 cells as a threshold and remove donors with less than 100 cells.  
All 10 donors below 100 are Healthy; at 50, only 2 would be lost.  
Since we have a 232 healthy donors, removing 10 will still leave us with 220 of them.  
50 is the sensitivity check

Final donor rule: a donor must be kept by the donor QC **and** have at least 100 cells that pass cell QC.
Each non-clean donor gets a short `reason`, so the README can list who was removed and why.

In [ ]:
min_cells = 100

donor_qc['n_pass'] = donor_qc['individual_id'].map(n_pass)
donor_qc['below_min'] = donor_qc['n_pass'] < min_cells
donor_qc['pass_donor'] = (donor_qc['donor_decision'] != 'drop') & ~donor_qc['below_min']

flag_names = {'low_counts': 'low counts', 'low_genes': 'low genes', 'high_mito': 'high mito'}

def donor_reason(row):
    parts = []
    if row['donor_decision'] == 'drop':
        parts.append('dropped: flagged by counts, genes and mito (5 MAD)')
    else:
        if row['donor_decision'] == 'keep_flagged':
            flags = [name for col, name in flag_names.items() if row[col]]
            parts.append('flagged (5 MAD): ' + ', '.join(flags))
        if row['below_min']:
            parts.append(f"excluded: only {int(row['n_pass'])} cells pass QC (< {min_cells})")
    return '; '.join(parts)

donor_qc['reason'] = donor_qc.apply(donor_reason, axis=1)

donor_qc.loc[donor_qc['reason'] != '',
            ['individual_id', 'health_category', 'donor_decision', 'n_pass', 'pass_donor', 'reason']]

In [ ]:
# donors below the minimum are removed from the analysis: update pass_qc on every cell
t_obs['pass_qc'] = t_obs['pass_qc'] & t_obs['individual_id'].map(
    donor_qc.set_index('individual_id')['pass_donor']).fillna(False).astype(bool)

print(f"donors kept: {donor_qc['pass_donor'].sum()} of {len(donor_qc)}")
print(f"cells passing QC: {t_obs['pass_qc'].sum():,} of {len(t_obs):,}")
donor_qc.groupby('health_category')['pass_donor'].agg(donors_kept='sum', donors_total='size')

Save the QC result so Stage 3 (pseudobulk) can load it without re-running this notebook.

In [ ]:
qc = (t_obs[['donor_ok', 'cell_ok', 'pass_qc', 'fail_counts', 'fail_genes', 'fail_mito']]
      .assign(mito_zero=t_obs['zero_mito'])
      .rename_axis('cell_id').reset_index())
qc.to_parquet(r'..\data\processed\qc_tcell.parquet', index=False)
donor_qc.to_parquet(r'..\data\processed\donor_qc_tcell.parquet', index=False)

# check: read back
print(pd.read_parquet(r'..\data\processed\qc_tcell.parquet').shape,
      pd.read_parquet(r'..\data\processed\donor_qc_tcell.parquet').shape)